In [2]:
import pandas as pd
import glob
import os

files = sorted(glob.glob("survey_results_public_*.csv"))

for path in files:
    year = path.split("_")[-1].replace(".csv", "")
    df = pd.read_csv(path, nrows=5)
    print(f"\n{'='*60}")
    print(f"YEAR: {year} | Shape: {pd.read_csv(path, nrows=0).shape[1]} cols | Sample rows: 5")
    print(f"{'='*60}")
    print(f"COLUMNS ({len(df.columns)}):")
    for col in df.columns:
        sample_vals = df[col].dropna().tolist()
        print(f"  {col}: {sample_vals[:2] if sample_vals else 'ALL NULL'}")


YEAR: 2021 | Shape: 48 cols | Sample rows: 5
COLUMNS (48):
  ResponseId: [1, 2]
  MainBranch: ['I am a developer by profession', 'I am a student who is learning to code']
  Employment: ['Independent contractor, freelancer, or self-employed', 'Student, full-time']
  Country: ['Slovakia', 'Netherlands']
  US_State: ALL NULL
  UK_Country: ['England']
  EdLevel: ['Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)', 'Bachelor’s degree (B.A., B.S., B.Eng., etc.)']
  Age1stCode: ['18 - 24 years', '11 - 17 years']
  LearnCode: ['Coding Bootcamp;Other online resources (ex: videos, blogs, etc)', 'Other online resources (ex: videos, blogs, etc);School']
  YearsCode: [7.0, 17.0]
  YearsCodePro: [10.0]
  DevType: ['Developer, mobile', 'Developer, front-end']
  OrgSize: ['20 to 99 employees', '100 to 499 employees']
  Currency: ['EUR European Euro', 'EUR European Euro']
  CompTotal: [4800.0]
  CompFreq: ['Monthly', 'Monthly']
  LanguageHaveWorkedWith: ['C++;HTML/CSS

In [3]:
import duckdb
con = duckdb.connect("survey.duckdb")

print(con.execute("""
    SELECT survey_year,
           COUNT(*) AS total,
           COUNT(salary_usd_yearly) AS has_salary,
           ROUND(MEDIAN(salary_usd_yearly)) AS median_salary,
           ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY salary_usd_yearly)) AS p25,
           ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY salary_usd_yearly)) AS p75
    FROM responses
    WHERE salary_usd_yearly BETWEEN 1000 AND 1000000
    GROUP BY 1
    ORDER BY 1
""").df().to_string(index=False))
con.close()

 survey_year  total  has_salary  median_salary     p25      p75
        2021  46013       46013        55776.0 27025.0  97288.0
        2022  36869       36869        66589.0 36000.0 116875.0
        2023  47275       47275        75000.0 45421.0 123000.0
        2024  22849       22849        66769.0 35890.0 109554.0
        2025  23177       23177        77000.0 41765.0 122976.0
